<a href="https://colab.research.google.com/github/CooL-Legend/CSCI544-DLM/blob/adi-e2/notebooks/gsm8k/E2_Experiment_SWD_Eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

# Experiment 2: Two-Step Lookahead MEDAL vs. AB-MEDAL on GSM8K with Stability-Weighted Decoding (SWD) — Evaluation (20 Canonical + 50 Extended Questions)

This notebook runs **Experiment 2 (E2)** on the GSM8K validation/evaluation pool with `LLaDA-8B-Instruct` using the **Two-Step SWD ($Q_{\mathrm{swd}}$)** reward function:
For parent node $s$, candidate child $c = s \oplus a_1$, and greedy virtual grandchild $g^* = c \oplus a_2^*$ ($a_2^* = A(c)[0]$), reward weights the grandchild token confidence $q^j_{t+2} = \max_v p^j_{t+2}(v)$ by trajectory distributional stability across positions $\mathcal{M}(g^*)$ remaining masked in $g^*$:
$$
Q_{\mathrm{swd}}(\text{Node}, \text{Action}) = \sum_{j \in \mathcal{M}(g^*)} q^j_{t+2} \cdot \exp\!\Big(-\beta\,\big(\mathrm{KL}^j(\text{Node}, \text{Child}) + \mathrm{KL}^j(\text{Child}, \text{GrandChild})\big)\Big)
$$
where $\mathrm{KL}^j(A, B) = D_{\mathrm{KL}}(p_A^j \,\|\, p_B^j)$ is the forward KL divergence along $s \xrightarrow{a_1} c \xrightarrow{a_2^*} g^*$ and $\beta = 1.0$.

### Dataset Split Organization:
1. **Canonical 20 Samples** (`sample_slice=(0, 20)`): The exact 20 evaluation questions from Experiment 1, preserving historical prefix ordering.
2. **Additional 50 Samples** (`sample_slice=(20, 70)`): 50 new validation questions sampled under the deterministic `sample_seed=1` order.
3. **Combined 70 Samples**: Full validation pool evaluation and paired statistical tests.

### Guardrails & Protocols:
- Capped at $4B$ simulations per question as an exploration runaway safeguard; actual NFE utilization is logged.
- Virtual grandchildren are evaluated and cached in `cached_analyses` but never attached to the tree or assigned GEN/CONT posteriors.
- **Compact Distribution Cache (0 Extra NFEs)**: Top-$512$ token probabilities plus exact residual uniform tail and exact $q^j_t = \max_v p^j_t(v)$ are cached per state directly from the single forward pass.
- **Empty Masks & 1-Step $R_{\mathrm{stability}}$ Fallback**: When $|\mathcal{M}(g^*)| = 0$ (terminal child or grandchild filling the last mask), $Q_{\mathrm{swd}} = 0.0$. When lookahead is unavailable (actionless non-terminal child or NFE boundary fallback), reward uses the 1-step stability objective $R_{\mathrm{stability}}(\text{Node}, \text{Child}) = \sum_{j \in \mathcal{M}(c)} q^j_{t+1} \cdot \exp(-\beta\,\mathrm{KL}^j(\text{Node}, \text{Child}))$.
- Non-mutating hypothetical boundary snapshots ensure standalone $B=256$ equivalence without altering live tree execution toward $B=512$.


## 1. Verify GPU Hardware
Verify CUDA GPU supporting bfloat16 (recommended: L4 or A100 with $\ge 22$ GiB VRAM).

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Select a CUDA GPU runtime in Colab (L4 or A100), then reconnect.')
gpu = torch.cuda.get_device_properties(0)
vram_gb = gpu.total_memory / (1024 ** 3)
print(f'GPU: {gpu.name} | Total VRAM: {vram_gb:.2f} GiB')


## 2. Repository Setup
Clone or update the GitHub repository on branch `adi-e2`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/CooL-Legend/CSCI544-DLM.git'
BRANCH = 'adi-e2'
REPO_DIR = Path('/content/CSCI544-DLM') if IN_COLAB else Path.cwd().resolve()

if IN_COLAB:
    if not (REPO_DIR / '.git').exists():
        token = os.environ.get('GITHUB_TOKEN', '')
        if not token:
            try:
                from google.colab import userdata
                token = userdata.get('GITHUB_TOKEN')
            except Exception:
                pass
        clone_url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL
        subprocess.run(['git', 'clone', '-b', BRANCH, clone_url, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', 'origin', BRANCH], check=True)
    os.chdir(REPO_DIR)
    if str(REPO_DIR / 'src') not in sys.path:
        sys.path.insert(0, str(REPO_DIR / 'src'))

print(f'Working directory: {Path.cwd()}')


## 3. Install Dependencies
Install pinned dependencies and editable package.

In [ ]:
import site
import subprocess
import sys

if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements/llada-colab.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
    import importlib
    importlib.invalidate_caches()
    user_site = site.getusersitepackages()
    if user_site not in sys.path:
        sys.path.append(user_site)
print('Dependencies installed successfully.')


## 4. Run Algorithmic Pre-Flight Unit Tests
Verify 2-step lookahead SWD KL-stability reward computation, 1-step $R_{\mathrm{stability}}$ fallback, state caching, and batched parity before loading weights.

In [ ]:
import subprocess
import sys

res = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/tasks/test_experiment_2.py', '-v'],
    capture_output=True, text=True
)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError('Pre-flight E2 unit tests failed!')
print('All E2 algorithmic unit tests passed.')


## 5. Mount Google Drive & Configure Cache
Mount Google Drive for persistent artifact and model caching.

In [ ]:
from pathlib import Path
import os

DRIVE_ROOT = Path('/content/drive/MyDrive/CSCI544-DLM')
OUTPUT_DIR = Path('results/e2_swd_eval')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = DRIVE_ROOT / 'results/e2_swd_eval'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    cache_dir = DRIVE_ROOT / 'cache/huggingface'
    cache_dir.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(cache_dir)
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(cache_dir)

print(f'Output directory: {OUTPUT_DIR.resolve()}')


## 6. Experiment Controls & Staged Model Loading
Configure budgets [256, 512], depth 10, batch size 2, SWD reward mode ($\beta=1.0$), and load `LLaDA-8B-Instruct` from staged NVMe storage.

In [ ]:
import shutil
from pathlib import Path
from dlm_project.models.profiles import get_profile
from dlm_project.tasks.gsm8k.experiment_2 import (
    CHECKPOINT_BUDGETS, MAX_SEARCH_DEPTH, MIN_CANDIDATES, context, load_model
)

STAGE = 'evaluation'
METHODS = ('medal', 'ab_medal')
BUDGETS = (256, 512)
REWARD_MODE = 'swd'
REWARD_LAMBDA = 1.0
REWARD_BETA = 1.0
SIM_CAP_MULTIPLIER = 4
# On NVIDIA L4 (22.4 GiB VRAM), BATCH_SIZE = 2 is recommended for stability;
# BATCH_SIZE = 4 is suited for A100 (40/80 GB) or High-RAM runtimes.
BATCH_SIZE = 2
RESUME = True

# Load base context to initialize model identity
base_ctx = context(
    stage='evaluation',
    methods=METHODS,
    budgets=BUDGETS,
    reward_mode=REWARD_MODE,
    reward_lambda=REWARD_LAMBDA,
    reward_beta=REWARD_BETA,
    sim_cap_multiplier=SIM_CAP_MULTIPLIER,
)

profile = get_profile('llada')
model_name = profile['repo_id'].split('/')[-1]
drive_model_dir = DRIVE_ROOT / 'models' / model_name
runtime_model_dir = Path('/content/models') / model_name

# Fast local NVMe cache in Colab runtime if available on Drive
if IN_COLAB and drive_model_dir.is_dir() and not (runtime_model_dir / 'config.json').is_file():
    print(f'Copying model from Google Drive to local fast runtime storage: {runtime_model_dir}...')
    shutil.copytree(drive_model_dir, runtime_model_dir)

MODEL_PATH = str(runtime_model_dir if runtime_model_dir.is_dir() else (drive_model_dir if drive_model_dir.is_dir() else profile['repo_id']))
print('Using model location:', MODEL_PATH)
print('Loading LLaDA-8B-Instruct wrapper on GPU in bfloat16...')
wrapper = load_model(base_ctx, model_path=MODEL_PATH, device='cuda', local_files_only=Path(MODEL_PATH).is_dir())
print(f'✓ Model loaded: {wrapper.identity["family"]} ({wrapper.identity["repo_id"]} @ {wrapper.identity["revision"]})')


## 7. Execute Part 1: Canonical Evaluation Set (Original 20 Questions)
Evaluates the exact 20 evaluation questions from Experiment 1 (`sample_slice=(0, 20)`).

In [ ]:
from time import perf_counter
from dlm_project.tasks.gsm8k.experiment_2 import run_e2_stage

ctx_20 = context(
    stage='evaluation',
    methods=METHODS,
    budgets=BUDGETS,
    sample_slice=(0, 20),
    reward_mode=REWARD_MODE,
    reward_lambda=REWARD_LAMBDA,
    reward_beta=REWARD_BETA,
    sim_cap_multiplier=SIM_CAP_MULTIPLIER,
)
print('Starting Part 1 (Canonical 20 Questions) with SWD reward...')
t0 = perf_counter()
q_rows_20, agg_rows_20 = run_e2_stage(
    wrapper,
    ctx_20,
    OUTPUT_DIR / 'canonical_20',
    methods=METHODS,
    budgets=BUDGETS,
    max_depth=MAX_SEARCH_DEPTH,
    num_candidates=MIN_CANDIDATES,
    resume=RESUME,
    batch_size=BATCH_SIZE,
)
print(f'Part 1 Complete in {(perf_counter() - t0) / 60:.2f} minutes.')


## 8. Execute Part 2: Extended Validation Set (Additional 50 Questions)
Evaluates the additional 50 validation questions (`sample_slice=(20, 70)`).

In [ ]:
ctx_50 = context(
    stage='evaluation',
    methods=METHODS,
    budgets=BUDGETS,
    sample_slice=(20, 70),
    reward_mode=REWARD_MODE,
    reward_lambda=REWARD_LAMBDA,
    reward_beta=REWARD_BETA,
    sim_cap_multiplier=SIM_CAP_MULTIPLIER,
)
print('Starting Part 2 (Additional 50 Questions) with SWD reward...')
t0 = perf_counter()
q_rows_50, agg_rows_50 = run_e2_stage(
    wrapper,
    ctx_50,
    OUTPUT_DIR / 'extended_50',
    methods=METHODS,
    budgets=BUDGETS,
    max_depth=MAX_SEARCH_DEPTH,
    num_candidates=MIN_CANDIDATES,
    resume=RESUME,
    batch_size=BATCH_SIZE,
)
print(f'Part 2 Complete in {(perf_counter() - t0) / 60:.2f} minutes.')


## 9. Results & Paired Comparisons (Canonical 20, Extended 50, and Combined 70)

In [ ]:
import pandas as pd
from IPython.display import display
from scripts.run_e2 import compute_paired_comparison
from dlm_project.experiments.sweep_report import write_csv

summaries_dir = OUTPUT_DIR / 'summaries'
summaries_dir.mkdir(parents=True, exist_ok=True)

print('=== Part 1: Canonical 20 Questions Paired Comparison ===')
paired_20 = compute_paired_comparison(q_rows_20, budgets=BUDGETS)
if paired_20:
    df_p20 = pd.DataFrame(paired_20)[['search_budget', 'paired_questions', 'medal_accuracy', 'ab_medal_accuracy', 'accuracy_delta_percent', 'ab_medal_only_correct', 'medal_only_correct', 'mcnemar_exact_p']]
    display(df_p20)
    write_csv(summaries_dir / 'e2_swd_canonical_20_paired_comparison.csv', paired_20)

print('=== Part 2: Additional 50 Questions Paired Comparison ===')
paired_50 = compute_paired_comparison(q_rows_50, budgets=BUDGETS)
if paired_50:
    df_p50 = pd.DataFrame(paired_50)[['search_budget', 'paired_questions', 'medal_accuracy', 'ab_medal_accuracy', 'accuracy_delta_percent', 'ab_medal_only_correct', 'medal_only_correct', 'mcnemar_exact_p']]
    display(df_p50)
    write_csv(summaries_dir / 'e2_swd_extended_50_paired_comparison.csv', paired_50)

print('=== Combined Pool (All 70 Questions) Paired Comparison ===')
combined_q_rows = q_rows_20 + q_rows_50
paired_combined = compute_paired_comparison(combined_q_rows, budgets=BUDGETS)
if paired_combined:
    df_pcomb = pd.DataFrame(paired_combined)[['search_budget', 'paired_questions', 'medal_accuracy', 'ab_medal_accuracy', 'accuracy_delta_percent', 'ab_medal_only_correct', 'medal_only_correct', 'mcnemar_exact_p']]
    display(df_pcomb)
    write_csv(summaries_dir / 'e2_swd_combined_70_paired_comparison.csv', paired_combined)
    write_csv(summaries_dir / 'e2_swd_evaluation_paired_comparison.csv', paired_combined)


## 10. Interactive Search Tree & Decision Trace Inspector
Inspect the search tree topology, candidate nodes, and decision trace for any evaluated question.

In [ ]:
from dlm_project.tasks.gsm8k.tree_viz import print_ascii_tree, summarize_decision_trace
import json

# Select a sample question to inspect from canonical_20
sample_ex_id = ctx_20['examples'][0]['example_id']
chk_dir = OUTPUT_DIR / 'canonical_20' / 'evaluation' / 'ab_medal' / sample_ex_id / 'checkpoints/256'
tree_json = chk_dir / 'tree.json'
metrics_json = chk_dir / 'metrics.json'

if tree_json.is_file():
    print(f'=== AB-MEDAL Search Tree at B=256 for {sample_ex_id} ===')
    print(print_ascii_tree(tree_json, max_depth=5, max_children_per_node=3))

if metrics_json.is_file():
    met = json.loads(metrics_json.read_text())
    print(f'=== Checkpoint Metrics ({sample_ex_id} @ B=256) ===')
    print(f'  Selected Node: {met["selected_node_id"]} | Depth: {met["selected_depth"]} | Q: {met["selected_q"]:.4f}')
    print(f'  Exact Match: {met["exact_match"]} | Pred: {met["predicted_answer"]} | Ref: {met["reference_answer"]}')
    print(f'  NFE Utilization: {met.get("nfe_utilization", 1.0)*100:.1f}% | Sims: {met.get("simulations_completed", "N/A")}/{met.get("simulation_cap", "N/A")}')


## 11. E3 Offline Metric Extraction & Publication Figures
Run offline tree allocation, branching dynamics, and structural analysis on saved evaluation checkpoints (zero GPU compute), and generate 6 publication figures.

In [ ]:
import shutil
from dlm_project.tasks.gsm8k.e3_analysis import analyze_e3_stage
from IPython.display import Image, display, Markdown

# Build combined trace directory linking canonical_20 and extended_50
combined_results_dir = OUTPUT_DIR / 'combined_70' / 'evaluation'
combined_results_dir.mkdir(parents=True, exist_ok=True)
for part_dir in [OUTPUT_DIR / 'canonical_20' / 'evaluation', OUTPUT_DIR / 'extended_50' / 'evaluation']:
    if part_dir.is_dir():
        for m in METHODS:
            m_src = part_dir / m
            if m_src.is_dir():
                m_dst = combined_results_dir / m
                m_dst.mkdir(parents=True, exist_ok=True)
                for q_dir in m_src.iterdir():
                    if q_dir.is_dir() and not (m_dst / q_dir.name).exists():
                        try:
                            (m_dst / q_dir.name).symlink_to(q_dir, target_is_directory=True)
                        except OSError:
                            shutil.copytree(q_dir, m_dst / q_dir.name)

# Select results directory: combined_70 if populated, else canonical_20
has_combined = any((combined_results_dir / m).is_dir() and any((combined_results_dir / m).iterdir()) for m in METHODS)
eval_results_dir = combined_results_dir if has_combined else (OUTPUT_DIR / 'canonical_20' / 'evaluation')
e3_output_dir = OUTPUT_DIR / ('combined_70' if has_combined else 'canonical_20') / 'summaries'
e3_output_dir.mkdir(parents=True, exist_ok=True)

print(f'Running E3 Offline Tree Analysis on {eval_results_dir}...')
e3_q, e3_agg = analyze_e3_stage(
    results_dir=eval_results_dir,
    output_dir=e3_output_dir,
    stage='evaluation',
    methods=tuple(METHODS),
    budgets=tuple(BUDGETS),
)
print(f'✓ E3 Analysis Completed successfully: {len(e3_q)} question outcomes evaluated.')

figures_dir = e3_output_dir / 'e3_figures'
fig_titles = {
    'fig1_nfe_by_depth': 'Figure 1: NFE Allocation by Tree Depth — MEDAL vs AB-MEDAL across budgets',
    'fig2_branching_factors': 'Figure 2: Branching Factor Distribution — Children per expanded parent',
    'fig3_candidate_density': 'Figure 3: Candidate Density vs Depth — Minimum-C availability and selection depth',
    'fig4_root_subtrees': 'Figure 4: Root-Subtree Compute Concentration — Search effort across root branches',
    'fig5_reward_redundancy': 'Figure 5: Reward and Search Redundancy — Negative IG rates and duplicate states',
    'fig6_accuracy_vs_budget': 'Figure 6: Accuracy vs Search Budget — Performance scaling curve',
}

for fig_key, title in fig_titles.items():
    png_path = figures_dir / f'{fig_key}.png'
    if png_path.is_file():
        display(Markdown(f'### {title}'))
        display(Image(filename=str(png_path), width=750))


## 12. Synchronize All Artifacts to Google Drive
Flush and sync all in-flight buffers to Google Drive.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.flush_and_unmount()
    print('Google Drive flushed successfully.')
